# Nachfragetreiber finden mit PROC GLMSELECT: Schrittweise Auswahl, LASSO und validierte Vorwärtsselektion

## Zusammenfassung

Ein Retail-Analytics-Team nutzt **PROC GLMSELECT**, um herauszufinden, welche Marketing- und Preishebel den wöchentlichen Absatz tatsächlich bewegen, und trennt so echte Nachfragetreiber vom Rauschen. Schrittweise Auswahl bewertet nach SBC, LASSO mit 5-facher Kreuzvalidierung und eine Holdout-validierte Vorwärtssuche liefern allesamt **dieselbe Menge echter Treiber** — eigener Preis, Wettbewerberpreis, Werbeausgaben, E-Mail-Volumen, Aktionen, Feiertage, die Region Northeast und der Kanal Online — und jeder der vier eingestreuten Scheinprädiktoren (`temp_f`, `noise1`, `noise2`, `noise3`) wird automatisch verworfen.

Auch bei den Effektgrößen stimmen die Methoden eng überein: Jede schätzt einen Eigenpreiseffekt nahe **-28 Einheiten pro Dollar** und einen Wettbewerberpreiseffekt nahe **+14** — genau die Substitutionsvorzeichen, die in die datengenerierende Gleichung eingebaut wurden. Die einzige Abweichung liegt am Rand: Die kreuzvalidierte LASSO behält zusätzlich einen kleinen, statistisch vernachlässigbaren `Region=Midwest`-Kontrast (Schätzwert -8,3 bei einem Standardfehler von 8,3), den sowohl die schrittweise als auch die Vorwärtsauswahl verwerfen. Eine Treiberliste, die drei unterschiedliche Auswahlphilosophien übersteht, ist weitaus vertrauenswürdiger als eine, die auf eine einzelne Regel zugeschnitten ist.

## Datenquellen

Alle Daten in diesem Notebook sind **synthetisch** und werden inline erzeugt (keine externen Dateien, Seed `20260531`). Sie bilden eine Saison Store-Week-Nachfragedaten für einen Konsumgüterhändler nach.

| Datensatz | Zeilen | Granularität | Schlüsselvariablen |
|---------|------|-------|---------------|
| `demand` | 100 | Store-Week | `units` (Zielgröße: wöchentlich verkaufte Einheiten); `price`, `competitor` (eigener & Konkurrenzpreis); `adspend`, `email` (Media-Druck); `promo`, `holiday` (Ereignis-Flags); `region`, `channel` (CLASS-Effekte); `temp_f`, `noise1`-`noise3` (Schein-/irrelevante Prädiktoren) |

`units` wird aus einer bekannten linearen Gleichung aufgebaut, sodass die "korrekte" Menge an Treibern überprüfbar ist; `temp_f` und die drei `noise`-Spalten tragen kein echtes Signal und dienen dazu zu testen, ob jede Auswahlmethode sie verwirft.

# Nachfragetreiber finden mit PROC GLMSELECT

Ein Category Manager hat Dutzende Kandidaten-Erklärgrößen für den wöchentlichen Absatz: eigener Preis, Wettbewerberpreis, Werbeausgaben, E-Mail-Volumen, Aktionen, Feiertage, Filialregion, Vertriebskanal, sogar das Wetter. Alle zusammen in eine einzige Regression zu werfen, lädt zu Overfitting und instabilen Koeffizienten ein. **PROC GLMSELECT** automatisiert die Suche nach einem sparsamen Modell und unterstützt schrittweise Auswahl, LASSO, Elastic Net und Least-Angle-Selektion mit eingebauter Kreuzvalidierung und Holdout-Partitionierung.

In diesem Notebook werden wir:

1. Ein realistisches synthetisches Store-Week-Nachfragepanel mit einer *bekannten* Menge echter Treiber erzeugen (zuzüglich absichtlicher Scheinvariablen).
2. Eine **schrittweise Auswahl** nach SBC durchführen.
3. Eine **LASSO**-Auswahl mit 5-facher Kreuzvalidierung durchführen.
4. Eine **Vorwärtsauswahl, validiert an einem 30%-Holdout**, durchführen.

Eine gute Auswahlmethode sollte die echten Treiber wiederfinden und das Rauschen verwerfen — schauen wir es uns an.

## 1. Ein synthetisches Nachfragepanel erzeugen

Die Zielgröße `units` wird aus einer expliziten linearen Gleichung konstruiert, sodass wir die Ground Truth kennen: Preis und Wettbewerberpreis, Werbeausgaben, E-Mail-Volumen, die Aktions- und Feiertags-Flags sowie Region- und Kanaleffekte spielen alle eine Rolle. Die Variablen `temp_f`, `noise1`, `noise2` und `noise3` sind reine Scheinprädiktoren ohne echten Zusammenhang zum Absatz. Ein `call streaminit`-Seed macht die Daten reproduzierbar.

In [1]:
/* ---------------------------------------------------------------
   Synthetisches Store-Week-Nachfragepanel fuer einen Konsumgueterhaendler.
   'units' folgt einer BEKANNTEN Gleichung; temp_f und noise1-3 sind Scheinpraediktoren.
   --------------------------------------------------------------- */
DATEN demand;
    AUFRUFEN streaminit(20260531);
    LÄNGE region $9 channel $8 promo $3;
    AUSFÜHRUNG store_week = 1 BIS 100;
        /* Regionsmischung */
        u1 = rand('uniform');
        WENN u1 < 0.34 DANN region = 'Northeast';
        SONST WENN u1 < 0.67 DANN region = 'Midwest';
        SONST region = 'South';

        /* Vertriebskanal */
        WENN rand('uniform') < 0.45 DANN channel = 'Store';
        SONST channel = 'Online';

        /* Preis- und Media-Treiber */
        price      = round(8  + rand('uniform') * 12, 0.01);
        competitor = round(8  + rand('uniform') * 12, 0.01);
        adspend    = round(rand('gamma', 2) * 500, 1);
        email      = round(rand('uniform') * 100, 1);

        /* Ereignis-Flags und ein irrelevanter Wetter-Scheinpraediktor */
        temp_f     = round(40 + rand('uniform') * 50, 0.1);
        holiday    = (rand('uniform') < 0.12);
        WENN rand('uniform') < 0.40 DANN promo = 'Yes';
        SONST promo = 'No';

        /* Reine Rausch-Praediktoren (kein echter Effekt) */
        noise1 = rand('normal');
        noise2 = rand('normal');
        noise3 = rand('normal');

        /* Woechentlich verkaufte Einheiten aus einer bekannten Strukturgleichung */
        units = 900
              - 28   * price
              + 14   * competitor
              + 0.06 * adspend
              + 1.2  * email
              + 55   * (promo = 'Yes')
              + 70   * holiday
              + 40   * (region = 'Northeast')
              + 25   * (channel = 'Online')
              + 30   * rand('normal');
        WENN units < 0 DANN units = 0;
        AUSGABE;
    ENDE;
    BEHALTEN region channel promo price competitor adspend email temp_f
         holiday noise1 noise2 noise3 units;
AUSFÜHREN;


NOTE: DATA demand


NOTE: Wrote demand (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.08 seconds
  cpu   0.08 seconds


## 2. Die Daten profilieren

Bevor wir modellieren, bestätigen wir, dass die Zielgröße und die wichtigsten stetigen Kandidaten auf sinnvollen Skalen liegen.

In [2]:
PROZEDUR MITTELWERTE DATEN=demand n mean std MIN MAX maxdec=1;
    VAR units price competitor adspend email;
    BEZEICHNUNG units="Verkaufte Einheiten" price="Eigener Preis" competitor="Wettbewerberpreis"
          adspend="Werbeausgaben" email="E-Mail-Volumen";
    TITEL "Wöchentliche Nachfrage und Kandidatentreiber";
AUSFÜHREN;

                                      Wöchentliche Nachfrage und Kandidatentreiber                                      

                                                  The MEANS Procedure

 Variable    Label                       N        Mean     Std Dev     Minimum     Maximum
 -----------------------------------------------------------------------------------------
 units       Verkaufte Einheiten       100       874.8       136.3       508.6      1162.3
 price       Eigener Preis             100        14.0         3.4         8.0        19.9
 competitor  Wettbewerberpreis         100        13.8         3.4         8.1        19.9
 adspend     Werbeausgaben             100       990.6       726.9        50.0      3358.0
 email       E-Mail-Volumen            100        45.5        26.4         0.0        99.0
 -----------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 3. Schrittweise Auswahl nach SBC

Die schrittweise Auswahl fügt Effekte abwechselnd hinzu und entfernt sie, hier beurteilt nach dem **Schwarz'schen Bayes-Kriterium (SBC)** sowohl für den Aufnahmetest (`select=sbc`) als auch für die finale Modellwahl (`choose=sbc`). SBC bestraft Komplexität stärker als AIC und bevorzugt schlankere Modelle.

Wichtige Anweisungen und Optionen:

- `CLASS region channel promo / param=reference` deklariert die kategorialen Prädiktoren mit Referenzzellen-Kodierung.
- `selection=stepwise(select=sbc choose=sbc)` steuert die Suche.
- `details=summary` gibt die schrittweise Auswahlübersicht aus; `stb` fügt standardisierte Koeffizienten hinzu, sodass Effekte auf unterschiedlichen Skalen vergleichbar werden.
- `output out=demand_scored p=predicted r=residual` speichert angepasste Werte und Residuen für die nachgelagerte Auswertung.

Lies die **Stepwise Selection Summary** als Suchprotokoll: Sie startet beim vollen 12-Effekt-Modell und *entfernt* Effekte nacheinander — `noise1`, `noise2`, `temp_f`, den `Region=Midwest`-Kontrast und `noise3` fallen der Reihe nach weg, weil jede Entfernung den SBC senkt. Was in der Tabelle **Parameter Estimates** übrig bleibt, ist das gewählte Modell.

In [3]:
PROZEDUR glmselect DATEN=demand seed=20260531;
    KLASSE region channel promo / PARAM=reference;
    MODELL units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=stepwise(AUSWÄHLEN=sbc choose=sbc)
          details=summary stb;
    BEZEICHNUNG units="Verkaufte Einheiten" region="Region" channel="Vertriebskanal" promo="Aktion"
          price="Eigener Preis" competitor="Wettbewerberpreis" adspend="Werbeausgaben"
          email="E-Mail-Volumen" temp_f="Temperatur (°F)" holiday="Feiertag"
          noise1="Störvariable 1" noise2="Störvariable 2" noise3="Störvariable 3";
    AUSGABE out=demand_scored p=predicted r=residual;
    TITEL "Schrittweise Auswahl der Nachfragetreiber (SBC)";
AUSFÜHREN;

                                      Wöchentliche Nachfrage und Kandidatentreiber                                      

The GLMSELECT Procedure


Dependent Variable: UNITS Verkaufte Einheiten


Number of Observations Used: 100

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                 Stepwise Selection Summary                                                  

    Step    Action            Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)
--------  --------  ----------------  -----------------  --------  --------  --------  --------  --------  --------  --------
       1   Removed    Störvariable 1                 12    0.9507    0.9439  707.0094  711.2420  713.1843  740.8766   12.0136


NOTE: PROC GLMSELECT data=demand

NOTE: The data set WORK.DEMAND_SCORED has 100 observations and 15 variables.
NOTE: PROC GLMSELECT statement used.


## 4. LASSO mit Kreuzvalidierung

Die LASSO schrumpft Koeffizienten in Richtung Null und führt Auswahl und Regularisierung gleichzeitig durch. Statt bei einem festen Kriterium zu stoppen, lassen wir eine **5-fache Kreuzvalidierung** den Punkt auf dem LASSO-Pfad mit dem besten Out-of-Sample-Fehler wählen.

- `selection=lasso(choose=cv stop=none)` verfolgt den vollständigen LASSO-Pfad und wählt den CV-optimalen Schritt.
- `cvmethod=random(5)` weist die Beobachtungen 5 zufälligen Folds zu.

Die **LASSO Selection Summary** zeigt die Reihenfolge, in der Effekte eintreten, während die Straffung nachlässt: zuerst `price`, dann `adspend`, `competitor`, die Region Northeast, `email`, `promo` und `holiday` — alle sieben echten Signale treten vor jedem Scheinprädiktor ein. Die Kreuzvalidierung wählt dann den Schritt mit dem niedrigsten Out-of-Sample-PRESS, und die resultierende Tabelle **Parameter Estimates** behält genau die echten Treiber (plus einen kleinen `Region=Midwest`-Term) und schließt `temp_f`, `noise1`, `noise2` und `noise3` aus, die erst ganz am Ende des Pfads eintreten.

In [4]:
PROZEDUR glmselect DATEN=demand seed=20260531;
    KLASSE region channel promo / PARAM=reference;
    MODELL units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=lasso(choose=cv STOPP=none)
          cvmethod=RANDOM(5);
    BEZEICHNUNG units="Verkaufte Einheiten" region="Region" channel="Vertriebskanal" promo="Aktion"
          price="Eigener Preis" competitor="Wettbewerberpreis" adspend="Werbeausgaben"
          email="E-Mail-Volumen" temp_f="Temperatur (°F)" holiday="Feiertag"
          noise1="Störvariable 1" noise2="Störvariable 2" noise3="Störvariable 3";
    TITEL "LASSO mit 5-facher Kreuzvalidierung";
AUSFÜHREN;

                                      Wöchentliche Nachfrage und Kandidatentreiber                                      

The GLMSELECT Procedure


Dependent Variable: UNITS Verkaufte Einheiten


Number of Observations Used: 100

  Cross Validation Information   

                   Item     Value
-----------------------  --------
Cross Validation Method    Random
        Number of Folds         5

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                          LASSO Selection Summary                                                          

    Step    Action             Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  -----------------  ------------


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 5. Vorwärtsauswahl, validiert an einem Holdout

Eine dritte, komplementäre Strategie: Das Modell wird per **Vorwärtsauswahl** aufgebaut (Effekte treten nur ein, verlassen das Modell nie), aber der Stopp erfolgt dort, wo die Leistung auf einer unabhängigen Holdout-Stichprobe am besten ist — ein direkter Schutz vor Overfitting.

- `partition fraction(validate=0.30)` reserviert zufällig 30 % der Zeilen für die Validierung, sodass 70 Trainings- und 30 Validierungsbeobachtungen übrig bleiben.
- `selection=forward(select=aic choose=validate)` fügt Effekte nach AIC hinzu, wählt das finale Modell aber nach dem Validierungsstichproben-Fehler.

Die Tabelle **Partition Fractions** bestätigt den 70/30-Split. Die Vorwärtsauswahl nimmt dann Effekte auf, bis sich der Validierungsfehler nicht mehr verbessert; die acht Effekte in der finalen Tabelle **Parameter Estimates** sind genau die echten Treiber, die vier Scheinprädiktoren werden nie zugelassen. Wenn drei auf unterschiedlichen Prinzipien aufbauende Methoden bei denselben Treibern landen, ist das Modell weitaus wahrscheinlicher real als ein Artefakt einer einzelnen Auswahlregel.

In [5]:
PROZEDUR glmselect DATEN=demand seed=20260531;
    KLASSE region channel promo / PARAM=reference;
    MODELL units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=forward(AUSWÄHLEN=aic choose=validate);
    BEZEICHNUNG units="Verkaufte Einheiten" region="Region" channel="Vertriebskanal" promo="Aktion"
          price="Eigener Preis" competitor="Wettbewerberpreis" adspend="Werbeausgaben"
          email="E-Mail-Volumen" temp_f="Temperatur (°F)" holiday="Feiertag"
          noise1="Störvariable 1" noise2="Störvariable 2" noise3="Störvariable 3";
    partition FRACTION(validate=0.30);
    TITEL "Vorwärtsauswahl validiert an einem 30%-Holdout";
AUSFÜHREN;

                                      Wöchentliche Nachfrage und Kandidatentreiber                                      

The GLMSELECT Procedure


Dependent Variable: UNITS Verkaufte Einheiten


Number of Observations Used: 70               
Number of Observations Used for Validation: 30

Partition Fractions 

      Role  Fraction
----------  --------
  Training    0.7000
Validation    0.3000
   Testing    0.0000

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                         Forward Selection Summary                                                         

    Step    Action             Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  --------------


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 6. Ergebnisse interpretieren

Alle drei Auswahlstrategien finden **dieselbe Menge echter Nachfragetreiber** und verwerfen jeden Scheinprädiktor. Direkt aus den Tabellen **Parameter Estimates** gelesen:

- **Der Preis** ist der dominante Treiber. Sein standardisierter Koeffizient im schrittweisen Modell beträgt **-0,70**, mit Abstand der größte Betrag, und die rohe Steigung liegt zwischen **-27,8** (schrittweise und LASSO) und **-28,6** (Vorwärtsauswahl) Einheiten pro Dollar — fast genau die -28, mit denen die Daten erzeugt wurden. Der **Wettbewerberpreis** bewegt die Nachfrage in die Gegenrichtung, bei rund **+14,4** über alle drei Anpassungen hinweg — das Substitutionsverhalten, das ein Category Manager erwartet.
- **Werbeausgaben** (etwa +0,062 Einheiten pro Dollar) und **E-Mail-Volumen** (etwa +1,2 Einheiten pro Aussendung) heben beide den Absatz und quantifizieren die Media-Reaktion.
- **Aktionen** und **Feiertage** tragen die größten Ereigniseffekte: Der `Promo=No`-Kontrast liegt bei etwa **-51 bis -57** relativ zu einer Aktionswoche, und der Feiertagseffekt liegt bei etwa **+66 bis +76** Einheiten. Die **Region Northeast** (etwa +49 bis +55) und der **Kanal Online** (etwa +24 bis +32) tragen positive Basiseffekte.
- Entscheidend: Jeder Scheinprädiktor — `temp_f`, `noise1`, `noise2`, `noise3` — wird von der schrittweisen und der Vorwärtsauswahl **verworfen** und ist aus dem CV-gewählten LASSO-Modell ausgeschlossen. Jede Methode hat das strukturelle Signal wiedergefunden und das Rauschen ignoriert.

Die drei Methoden sind nicht Byte für Byte identisch: Die schrittweise Auswahl (SBC) und die Holdout-validierte Vorwärtssuche einigen sich auf dieselben acht Effekte, während die kreuzvalidierte LASSO zusätzlich einen kleinen `Region=Midwest`-Kontrast behält (-8,3, Standardfehler 8,3) — ein Unterschied im Rauschbereich und keine inhaltliche Abweichung. Dass eine Treiberliste die schrittweise SBC-Auswahl, die kreuzvalidierte LASSO und die Holdout-validierte Vorwärtsauswahl übersteht, ist die eigentliche Erkenntnis: Ein Befund, der drei unterschiedliche Auswahlphilosophien übersteht, ist weitaus glaubwürdiger als einer, der auf ein einzelnes Kriterium zugeschnitten ist. Mit `OUTPUT OUT=demand_scored`, das angepasste Werte und Residuen erfasst, lässt sich derselbe Workflow natürlich auf die Bewertung des geplanten Preis- und Aktionskalenders des nächsten Quartals ausweiten.